In [1]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

connected to port: 61585


In [2]:
# path = "export.conllu"
path = "test.conllu"
grewpy.set_config('sud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

In [3]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])

In [4]:
all_matches

{'na': [{'sent_id': 'JOS_12_How-To-Prepare-Gote-Soup_MG__4',
   'matching': {'nodes': {'X': '2'}, 'edges': {}}}],
 'minute': [{'sent_id': 'JOS_12_How-To-Prepare-Gote-Soup_MG__4',
   'matching': {'nodes': {'X': '6'}, 'edges': {}}}],
 'just': [{'sent_id': 'JOS_12_How-To-Prepare-Gote-Soup_MG__4',
   'matching': {'nodes': {'X': '3'}, 'edges': {}}}],
 'five': [{'sent_id': 'JOS_12_How-To-Prepare-Gote-Soup_MG__4',
   'matching': {'nodes': {'X': '5'}, 'edges': {}}}],
 'about': [{'sent_id': 'JOS_12_How-To-Prepare-Gote-Soup_MG__4',
   'matching': {'nodes': {'X': '4'}, 'edges': {}}}]}

In [5]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 0:
        matches[key] = value
print(len(matches))

5


In [6]:
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)

In [7]:
with open("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

data = { k : list() for k in match_upos }
for node, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features_prosody(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[node].append(formatted_features)

In [8]:
data

{('na',
  'PART'): [['node:X:own:AlignBegin=27760',
   'node:X:own:AlignEnd=27930',
   'node:X:own:Gloss=be',
   'node:X:own:PartType=Cop',
   'node:X:own:UtteranceMeanF0=123.239',
   'node:X:own:phoneticform=na',
   'node:X:own:word.MeanF0_Avg=121.791',
   'node:X:own:word.MeanF0_Range=0.0',
   'node:X:own:word.MeanF0_Slope=0.0',
   'node:X:own:word.MeanF0_Max=121.791',
   'node:X:own:word.MeanF0Normalized_Avg=0.562',
   'node:X:own:word.MeanF0Normalized_Range=0.0',
   'node:X:own:word.MeanF0Normalized_Slope=0.0',
   'node:X:own:word.MeanF0Normalized_Max=0.562',
   'node:X:own:word.AvgAmplitude_Avg=64.581',
   'node:X:own:word.AvgAmplitude_Range=0.0',
   'node:X:own:word.AvgAmplitude_Slope=0.0',
   'node:X:own:word.AvgAmplitude_Max=64.581',
   'node:X:own:word.AvgAmplitudeNormalized_Avg=0.722',
   'node:X:own:word.AvgAmplitudeNormalized_Range=0.0',
   'node:X:own:word.AvgAmplitudeNormalized_Slope=0.0',
   'node:X:own:word.AvgAmplitudeNormalized_Max=0.722',
   'node:X:own:word.MaxAmpli

In [9]:
unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [9]:
data[('just', 'ADV')]

[['node:X:own:AlignBegin=27930',
  'node:X:own:AlignEnd=28160',
  'node:X:own:Gloss=just',
  'node:X:own:phoneticform=dZOs',
  'node:X:own:rel_shallow=mod',
  'node:X:parent:position=after',
  'node:X:parent:AlignBegin=28160',
  'node:X:parent:AlignEnd=28420',
  'node:X:parent:Gloss=about',
  'node:X:parent:phoneticform=abat',
  'node:X:parent:upos=ADP',
  'node:X:prev:AlignBegin=27760',
  'node:X:prev:AlignEnd=27930',
  'node:X:prev:Gloss=be',
  'node:X:prev:PartType=Cop',
  'node:X:prev:UtteranceMeanF0=123.239',
  'node:X:prev:phoneticform=na',
  'node:X:prev:upos=PART',
  'node:X:next:AlignBegin=28160',
  'node:X:next:AlignEnd=28420',
  'node:X:next:Gloss=about',
  'node:X:next:phoneticform=abat',
  'node:X:next:upos=ADP',
  'node:X:child:AlignBegin=27930',
  'node:X:child:AlignEnd=28060',
  'node:X:child:AvgAmplitude=69.49',
  'node:X:child:AvgAmplitudeGlobalZscore=0.703',
  'node:X:child:AvgAmplitudeLocalZscore=1.126',
  'node:X:child:AvgAmplitudeNormalized=1',
  'node:X:child:Avg

In [10]:
# test_unique_features = ["node:X:next:AlignBegin=129510", "node:X:prev:upos=NOUN"]
import re
new_unique_features = []
for feat in unique_features:
    if re.search("phoneticform", feat):
        continue
    elif re.search("Gloss=", feat):
        continue
    elif re.search("Nucleus", feat):
        continue
    elif re.search(":Onset", feat):
        continue
    elif re.search("SylForm", feat):
        continue
    elif re.search(r"=$", feat):
        continue

    if re.search(r"\d.?\d?$", feat):
        numberless_feat = feat.split("=")[0]
        print(f"{feat} -> {numberless_feat}")
        new_unique_features.append(numberless_feat)
    else:
        new_unique_features.append(feat)
    
new_unique_features = list(set(new_unique_features))
len(new_unique_features)

node:X:child:all.AvgHeightGlo=H_Ratio=0.3333333333333333 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightGlo=H_Ratio=0.5 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightGlo=L_Ratio=0.3333333333333333 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightGlo=M_Ratio=0.3333333333333333 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightGlo=M_Ratio=0.5 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightGlo=M_Ratio=1.0 -> node:X:child:all.AvgHeightGlo
node:X:child:all.AvgHeightLoc=H_Ratio=0.3333333333333333 -> node:X:child:all.AvgHeightLoc
node:X:child:all.AvgHeightLoc=H_Ratio=0.5 -> node:X:child:all.AvgHeightLoc
node:X:child:all.AvgHeightLoc=M_Ratio=0.5 -> node:X:child:all.AvgHeightLoc
node:X:child:all.AvgHeightLoc=M_Ratio=0.6666666666666666 -> node:X:child:all.AvgHeightLoc
node:X:child:all.AvgHeightLoc=M_Ratio=1.0 -> node:X:child:all.AvgHeightLoc
node:X:child:all.Avg_AvgAmplitude=62.243 -> node:X:child:all.Avg_AvgAmplitude
node:X:child:all.Avg_A

402

In [20]:
new_unique_features

['node:X:own:word.AvgAmplitude_Avg',
 'node:X:next:upos=ADP',
 'node:X:next:syl_last.Duration',
 'node:X:prev:syl1.Coda=None',
 'node:X:parent:syl1.MeanF0',
 'node:X:own:syl1.Coda=None',
 'node:X:prev:word.MeanF0Normalized_Range',
 'node:X:parent:word.MaxAmplitude_Max',
 'node:X:prev:syl_last.Slope=Fall',
 'node:X:prev:syl_last.SlopeLoc=Rise',
 'node:X:child:all.Avg_MeanF0',
 'node:X:child:all.Slope',
 'node:X:own:syl1.MeanF0',
 'node:X:own:word.SemitonesFromUtteranceMean_Max',
 'node:X:child:all.Avg_SemitonesFromUtteranceMean',
 'node:X:prev:syl1.PitchRangeGlo=L',
 'node:X:parent:syl_last.MeanF0Normalized',
 'node:X:own:syl1.Onset=m',
 'node:X:next:word.MeanF0_Slope',
 'node:X:own:syl1.Duration',
 'node:X:own:word.MeanF0_Max',
 'node:X:parent:word.SemitonesFromUtteranceMean_Range',
 'node:X:own:syl1.AvgHeightLoc=H',
 'node:X:child:all.Avg_MaxAmplitude',
 'node:X:own:syl1.MeanF0Normalized',
 'node:X:prev:syl1.Onset=dZ',
 'node:X:own:syl1.PitchRangeGlo=L',
 'node:X:next:syl_last.Slope=R

In [23]:
def prune_features(features):
    """
    Removes redundant, leaking, or overly sparse features.
    """
    clean_features = []
    
    # 1. Define patterns to REMOVE
    # We want Normalized only, so we remove the raw versions if they aren't 'Normalized'
    # But we must be careful not to remove 'Normalized' itself.
    raw_acoustic_keywords = ["MeanF0", "AvgAmplitude", "MaxAmplitude", "Duration"]
    
    for key in features:
        # Convert tuple key to string for easier checking if needed, 
        # but your keys seem to be tuples: ('node', 'X', 'own', 'word.MeanF0_Avg')
        
        # Get the feature name part (usually the last element or the one with the dot)
        # Based on your list: key might be a string "node:X:..." or a tuple.
        # Assuming string for this check:
        feat_str = str(key) 

        # --- FILTER 1: Timestamps (Data Leak) ---
        if "AlignBegin" in feat_str or "AlignEnd" in feat_str:
            continue

        # --- FILTER 2: Specific Phonemes (Sparsity) ---
        # We only want to know if Coda exists (None vs Not None), not if it is 'f' or 't'
        if "Onset=" in feat_str or ("Coda=" in feat_str and "Coda=None" not in feat_str):
            continue

        # --- FILTER 3: Raw Acoustics (Redundancy) ---
        # If it has "MeanF0" but NOT "Normalized" -> Drop it
        is_raw_acoustic = False
        for raw_kw in raw_acoustic_keywords:
            # Check if it contains the raw keyword (e.g. MeanF0)
            if raw_kw in feat_str:
                # But keep it if it is the Normalized version
                if "Normalized" not in feat_str:
                    is_raw_acoustic = True
                    break
        
        if is_raw_acoustic:
            continue

        # --- FILTER 4: "Average of Averages" (Redundancy) ---
        # Keep MaxAmplitude (better for stress), Drop AvgAmplitude
        if "AvgAmplitude" in feat_str:
            continue

        # If it passed all filters, keep it
        clean_features.append(key)

    return clean_features

In [24]:
new_unique_features = prune_features(new_unique_features)
new_unique_features

['node:X:next:upos=ADP',
 'node:X:prev:syl1.Coda=None',
 'node:X:own:syl1.Coda=None',
 'node:X:prev:word.MeanF0Normalized_Range',
 'node:X:prev:syl_last.Slope=Fall',
 'node:X:prev:syl_last.SlopeLoc=Rise',
 'node:X:child:all.Slope',
 'node:X:own:word.SemitonesFromUtteranceMean_Max',
 'node:X:child:all.Avg_SemitonesFromUtteranceMean',
 'node:X:prev:syl1.PitchRangeGlo=L',
 'node:X:parent:syl_last.MeanF0Normalized',
 'node:X:parent:word.SemitonesFromUtteranceMean_Range',
 'node:X:own:syl1.AvgHeightLoc=H',
 'node:X:own:syl1.MeanF0Normalized',
 'node:X:own:syl1.PitchRangeGlo=L',
 'node:X:next:syl_last.Slope=Rise',
 'node:X:prev:syl_last.PitchRangeGlo=L',
 'node:X:parent:syl_last.Slope=Flat',
 'node:X:next:syl_last.SlopeGlo=Flat',
 'node:X:next:word.SemitonesFromUtteranceMean_Range',
 'node:X:prev:NumType=Card',
 'node:X:prev:upos=ADV',
 'node:X:own:word.MeanF0Normalized_Avg',
 'node:X:prev:word.TotalDurationNormalized',
 'node:X:prev:syl_last.SlopeGlo=Flat',
 'node:X:prev:syl_last.DurationNo

In [28]:
def prune_final_list(features):
    clean_features = []
    for key in features:
        feat_str = str(key)
        
        # 1. Drop Semitones (Redundant with Normalized F0)
        if "SemitonesFromUtteranceMean" in feat_str: continue
            
        # 2. Drop Categorical Prosody (Redundant with Continuous Stats)
        # We look for features that have exact values like =Fall, =Rise, =H, =L
        if any(x in feat_str for x in ["Slope=", "SlopeLoc=", "SlopeGlo=", 
                                       "AvgHeight", "PitchRange"]):
            continue
            
        # 3. Drop Syllable details for Neighbors (Too granular)
        # If it is 'prev' or 'next', ONLY allow 'word.' features (not syl1/syl_last)
        if ("prev:" in feat_str or "next:" in feat_str) and ("syl1" in feat_str or "syl_last" in feat_str):
            continue
            
        clean_features.append(key)
        
    return clean_features

In [29]:
new_unique_features = prune_final_list(new_unique_features)
new_unique_features

['node:X:next:upos=ADP',
 'node:X:own:syl1.Coda=None',
 'node:X:prev:word.MeanF0Normalized_Range',
 'node:X:child:all.Slope',
 'node:X:parent:syl_last.MeanF0Normalized',
 'node:X:own:syl1.MeanF0Normalized',
 'node:X:prev:NumType=Card',
 'node:X:prev:upos=ADV',
 'node:X:own:word.MeanF0Normalized_Avg',
 'node:X:prev:word.TotalDurationNormalized',
 'node:X:parent:word.MaxAmplitudeNormalized_Max',
 'node:X:prev:upos=PUNCT',
 'node:X:prev:word.MeanF0Normalized_Avg',
 'node:X:prev:word.MaxAmplitudeNormalized_Slope',
 'node:X:prev:word.MaxAmplitudeNormalized_Avg',
 'node:X:next:word.MaxAmplitudeNormalized_Max',
 'node:X:next:word.MeanF0Normalized_Range',
 'node:X:parent:syl1.Coda=None',
 'node:X:parent:syl1.MaxAmplitudeNormalized',
 'node:X:own:syl_last.MaxAmplitudeNormalized',
 'node:X:own:word.MaxAmplitudeNormalized_Max',
 'node:X:parent:upos=ADP',
 'node:X:own:rel_shallow=comp:obj',
 'node:X:parent:word.MeanF0Normalized_Range',
 'node:X:child:all.Onset',
 'node:X:own:rel_shallow=det:num',


In [30]:
len(new_unique_features)

87

In [31]:
idx2feature = {i : feat for i, feat in enumerate(new_unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}

In [32]:
idx2feature

{0: 'node:X:next:upos=ADP',
 1: 'node:X:own:syl1.Coda=None',
 2: 'node:X:prev:word.MeanF0Normalized_Range',
 3: 'node:X:child:all.Slope',
 4: 'node:X:parent:syl_last.MeanF0Normalized',
 5: 'node:X:own:syl1.MeanF0Normalized',
 6: 'node:X:prev:NumType=Card',
 7: 'node:X:prev:upos=ADV',
 8: 'node:X:own:word.MeanF0Normalized_Avg',
 9: 'node:X:prev:word.TotalDurationNormalized',
 10: 'node:X:parent:word.MaxAmplitudeNormalized_Max',
 11: 'node:X:prev:upos=PUNCT',
 12: 'node:X:prev:word.MeanF0Normalized_Avg',
 13: 'node:X:prev:word.MaxAmplitudeNormalized_Slope',
 14: 'node:X:prev:word.MaxAmplitudeNormalized_Avg',
 15: 'node:X:next:word.MaxAmplitudeNormalized_Max',
 16: 'node:X:next:word.MeanF0Normalized_Range',
 17: 'node:X:parent:syl1.Coda=None',
 18: 'node:X:parent:syl1.MaxAmplitudeNormalized',
 19: 'node:X:own:syl_last.MaxAmplitudeNormalized',
 20: 'node:X:own:word.MaxAmplitudeNormalized_Max',
 21: 'node:X:parent:upos=ADP',
 22: 'node:X:own:rel_shallow=comp:obj',
 23: 'node:X:parent:word.M

In [27]:
idx2adv

{0: ('about', 'ADP'),
 1: ('five', 'NUM'),
 2: ('just', 'ADV'),
 3: ('minute', 'NOUN'),
 4: ('na', 'PART')}

In [22]:
import re

txt = 'node:X:child:MeanF0Normalized=0.562'
x = re.search("\d", txt)
if re.search(r"\d.?\d$", txt):
	print("hello")
# txt.split("=")[0]

hello


In [ ]:
l = ['node:X:child:Glo=hm']
if "Glo" in l:
    print("hi")

In [28]:
import re
from collections import defaultdict

X = np.zeros((len(data.keys()), len(new_unique_features)))
n_samples_total = len(matches)
for adv, samples in data.items():
    row_idx = adv2idx[adv]
    numerical_feats = defaultdict(float)
    for m in samples:
        for feature in m:
            # print("Currently looking at ", feature)
            try:
                feature_name, value = feature.split("=")
            except ValueError:
                print(f"Could not split feature {feature}")
                continue
            if feature_name in new_unique_features:
                numerical_feats[feature_name] += float(value)
            elif feature in new_unique_features:
                X[row_idx, feature2idx[feature]] += 1
            # else:
                # print(f"Feature {feature} not in the given list")
    X[row_idx] = X[row_idx] / n_samples_total
    n_samples_adv = len(samples)
    for feat_name, total_value in numerical_feats.items():
        if feat_name in feature2idx:
            col_idx = feature2idx[feat_name]
            if X[row_idx, col_idx] == 0:
                X[row_idx, col_idx] = total_value / n_samples_adv

print(f"{X.shape=}")

X.shape=(5, 96)


In [30]:
X

array([[ 2.79300e+04,  0.00000e+00,  2.00000e-01,  0.00000e+00,
         0.00000e+00,  0.00000e+00,  2.84200e+04,  2.00000e-01,
         2.00000e-01,  1.34400e+00,  0.00000e+00,  8.94000e-01,
         2.00000e-01,  0.00000e+00,  3.00000e+00,  0.00000e+00,
        -2.03000e-01,  0.00000e+00,  1.37698e+02,  2.81600e+04,
         2.00000e-01,  2.00000e-01,  2.79300e+04,  0.00000e+00,
         2.00000e-01,  1.02300e+00,  0.00000e+00,  3.60000e+02,
         1.23239e+02,  0.00000e+00,  0.00000e+00,  0.00000e+00,
         0.00000e+00,  2.00600e+00,  1.25400e+00,  2.00000e-01,
         0.00000e+00,  2.00000e-01,  2.00000e-01,  2.00000e-01,
         0.00000e+00,  0.00000e+00,  2.00000e-01,  0.00000e+00,
         2.00000e-01,  2.00000e-01, -2.03000e-01,  1.12980e+05,
         2.00000e-01,  0.00000e+00, -6.14000e-01,  2.00000e-01,
         0.00000e+00,  2.00000e-01,  0.00000e+00,  0.00000e+00,
         1.14010e+05,  0.00000e+00,  5.56000e-01,  0.00000e+00,
         1.25046e+02,  2.00000e-01,  2.0

In [32]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

corpus = tod.corpus.CorpusProsody(
    # treebank_path="/Users/madalina/Downloads/pSUD_Naija-NSC",
    treebank_path="test.conllu",
    grew_pattern="pattern { X[upos<>PUNCT] }",
    patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt",
    min_occurrences=0,
)

In [33]:
# check the feature matrix ((are there other values than 0?))
import numpy as np
np.sum(corpus.feature_matrix)

8533203.338785715

In [ ]:
clustering = tod.clustering.SparseKMeans(corpus=corpus, k=10, top_n_features=10, write_to_file=True)
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)
fig.show()
# fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/28. naija_prosody/sparse_kmeans_naija_prosody.html")